In [ ]:
import numpy as np
import scipy.io
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Settings

In [ ]:
SEQ_LENGTH = 20
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.2
LEARNING_RATE = 0.001
BATCH_SIZE = 32
EPOCHS = 120

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load data

In [ ]:
train_mat = scipy.io.loadmat("Xtrain.mat")
test_mat = scipy.io.loadmat("Xtest.mat")

print(train_mat.keys())
print(test_mat.keys())

train_data = train_mat["Xtrain"].flatten()
test_data = test_mat["Xtest"].flatten()

# Scale fit only on train data

In [ ]:
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(
    train_data.reshape(-1, 1)
)

test_scaled = scaler.transform(
    test_data.reshape(-1, 1)
)

In [ ]:
split_idx = int(len(train_scaled) * 0.8)

train_part = train_scaled[:split_idx]
val_part = train_scaled[split_idx:]

def create_sequences(data, seq_length):

    X = []
    y = []

    for i in range(len(data) - seq_length):

        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])

    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_part, SEQ_LENGTH)
X_val, y_val = create_sequences(val_part, SEQ_LENGTH)

In [ ]:
class TimeSeriesDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TimeSeriesDataset(X_train, y_train)
val_dataset = TimeSeriesDataset(X_val, y_val)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# LSTM model

In [ ]:
class LSTMModel(nn.Module):

    def __init__(self,
                 input_size=1,
                 hidden_size=64,
                 num_layers=2,
                 dropout=0.2):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):

        out, _ = self.lstm(x)

        # Take last timestep
        out = out[:, -1, :]

        out = self.fc(out)

        return out

model = LSTMModel(
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

# loss + optim

In [ ]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

# Training with ES

In [ ]:
train_losses = []
val_losses = []

best_val_loss = float("inf")
patience = 15
counter = 0
best_model_state = None

for epoch in range(EPOCHS):
    model.train()

    running_train_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)

    model.eval()

    running_val_loss = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            running_val_loss += loss.item()

    avg_val_loss = running_val_loss / len(val_loader)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0
        best_model_state = model.state_dict()
    else:
        counter += 1

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {avg_train_loss:.6f} | "
        f"Val Loss: {avg_val_loss:.6f}"
    )

    if counter >= patience:
        print(f"Early stopping at epoch {epoch+1} with best val loss {best_val_loss:.6f}")
        break

if best_model_state is not None:
    model.load_state_dict(best_model_state)

# Plot training

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend()
plt.show()

# retraining on full train set

## After choosing hyperparam

In [ ]:
X_full, y_full = create_sequences(train_scaled, SEQ_LENGTH)

full_dataset = TimeSeriesDataset(X_full, y_full)

full_loader = DataLoader(
    full_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

final_model = LSTMModel(
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

final_optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=LEARNING_RATE
)

# Retrain on ALL train data

for epoch in range(EPOCHS):

    final_model.train()

    for X_batch, y_batch in full_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        final_optimizer.zero_grad()

        outputs = final_model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        final_optimizer.step()

# Test eval

In [ ]:
X_test, y_test = create_sequences(test_scaled, SEQ_LENGTH)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
).to(device)

final_model.eval()

with torch.no_grad():

    predictions = final_model(X_test_tensor)

predictions = predictions.cpu().numpy()

predictions_inverse = scaler.inverse_transform(predictions)

y_test_inverse = scaler.inverse_transform(y_test)

In [ ]:
mae = mean_absolute_error(
    y_test_inverse,
    predictions_inverse
)

mse = mean_squared_error(
    y_test_inverse,
    predictions_inverse
)

print("\nFINAL TEST RESULTS")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", np.sqrt(mse))
print("R2:", 1 - mse / np.var(y_test_inverse))


# plot test

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    y_test_inverse,
    label="Real Values"
)

plt.plot(
    predictions_inverse,
    label="Predictions"
)

plt.title("LSTM Predictions vs Real Test Values")

plt.xlabel("Time")
plt.ylabel("Signal")

plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(predictions_inverse.flatten(), color="orange")
plt.title("Full Predicted Set")
plt.xlabel("Time")
plt.ylabel("Signal")

plt.subplot(1, 2, 2)
plt.plot(y_test_inverse.flatten(), color="blue")
plt.title("Actual Test Values")
plt.xlabel("Time")
plt.ylabel("Signal")

plt.tight_layout()
plt.show()